In [ ]:
# Fresh Start - Run this cell first in every new Colab session

import shutil
import sys
from pathlib import Path
from google.colab import drive

print("🔄 Starting fresh session...")

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Create clean symlink
!rm -f /content/Japanese_RAG_Production
!ln -s "/content/drive/MyDrive/Japanese_RAG_Production" /content/Japanese_RAG_Production

# Set Python paths
PROJECT_ROOT = Path("/content/Japanese_RAG_Production")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Clear old ChromaDB to avoid dimension/permission issues
CHROMA_PATH = Path("/content/chroma_db")
if CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)
    print("✅ Old ChromaDB cleared")

print("✅ Fresh start completed")

🔄 Starting fresh session...
Mounted at /content/drive
✅ Fresh start completed successfully!



In [ ]:
!pip install -q pymupdf sentence-transformers chromadb groq python-dotenv tqdm fugashi unidic-lite
print("✅ Packages installed")

In [ ]:
from src.ingestion import DocumentIngestion
from src.chunking import JapaneseChunker
from src.embedding_store import EmbeddingStore
from src.retrieval import Retriever
from src.config import config

print("🚀 Running full pipeline test...\n")

# 1. Load documents
ingestor = DocumentIngestion()
docs = ingestor.load_all_documents(config.DATA_SAMPLE)
print(f"✅ Loaded {len(docs)} documents")

# 2. Chunk documents
chunker = JapaneseChunker()
all_chunks = []
for doc in docs:
    chunks = chunker.chunk_text(doc["content"], doc["filename"])
    all_chunks.extend(chunks)
print(f"✅ Created {len(all_chunks)} chunks")

# 3. Embed & Store
store = EmbeddingStore()
store.add_documents(all_chunks)

# 4. Test retrieval
retriever = Retriever()
query = "楽天の2025年度の連結Non-GAAP営業利益はいくらでしたか？"
results = retriever.retrieve(query)

print(f"\n✅ Retrieval test completed")
print(f"Query: {query}")
print(f"Chunks retrieved: {len(results)}")

if results:
    print("\nTop retrieved chunk preview:")
    print(results[0]["content"][:300] + "...")

print("\n✅ Full pipeline test completed successfully!")

✅ Loaded: TDK.pdf (4558 characters)
✅ Loaded: Rakuten 統合報告書 2025.pdf (127683 characters)
✅ Loaded: ジェトロ対日投資報告.pdf (67429 characters)
✅ Loaded: ＧＸ需要創出に向けた研究会.pdf (18560 characters)
✅ Loaded: 通商白書1-96.pdf (109048 characters)

📊 Total documents loaded: 5
✅ Created 11 chunks from TDK.pdf
✅ Created 251 chunks from Rakuten 統合報告書 2025.pdf
✅ Created 140 chunks from ジェトロ対日投資報告.pdf
✅ Created 36 chunks from ＧＸ需要創出に向けた研究会.pdf
✅ Created 213 chunks from 通商白書1-96.pdf


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

✅ Embedding model loaded: BAAI/bge-m3


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

✅ Added 651 chunks to vector store


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Embedding model loaded: BAAI/bge-m3

📊 Evaluation Results:
   Query: TDKのAI/MLエンジニアの主な業務は何ですか？
   Chunks retrieved: 5
   Keyword hit rate: 100.00%
   Average chunk length: 392 chars

✅ Pipeline + Evaluation completed!
